# Handwriting-aware Bigram tokenizer

This notebook is the handwriting-aware counterpart to `train_tokenizers.ipynb`. The underlying IAM+READ handwriting-bigram model is already trained and frozen in the DTLR project; this notebook does **not** learn bigrams from OnHW labels. It authenticates the pinned external model, constructs the deterministic leakage-safe OnHW adapter, verifies the canonical artifact, and checks label compatibility.

The primary artifact must always contain 419 CTC classes: blank ID 0, 59 fixed OnHW characters, and 359 IAM+READ-derived bigrams.

In [ ]:
import hashlib
import json
import unicodedata
from pathlib import Path

from tva.handwriting_bigram_adapter import (
    ADAPTER_BIGRAM_COUNT,
    ADAPTER_POLICY_ID,
    ADAPTER_SHA256,
    ADAPTER_SIZE,
    SOURCE_SHA256,
    build_frozen_onhw_adapter,
    canonical_json_bytes,
    write_adapter,
)
from tva.tokenizers import get_tokenizer

REPO_ROOT = Path.cwd().resolve()
assert (REPO_ROOT / 'tva').is_dir(), 'Run this notebook from the TVA repository root.'

## Paths and write policy

Change `SOURCE_PATH` only if the frozen DTLR output is stored elsewhere. The builder rejects any source whose complete SHA-256 or schema differs from the audited IAM+READ model. Existing canonical output is verified rather than overwritten by default.

In [ ]:
SOURCE_PATH = Path('/home/artellisys/dtlr-output/combined-iam-read-v1/model.json')
OUTPUT_PATH = REPO_ROOT / 'artifacts/tokenizers/onhw_words500_rh_iam_read_v1.json'
WRITE_CANONICAL_ARTIFACT = False

assert SOURCE_PATH.is_file(), f'Frozen DTLR source not found: {SOURCE_PATH}'
source_digest = hashlib.sha256(SOURCE_PATH.read_bytes()).hexdigest()
assert source_digest == SOURCE_SHA256
print('Authenticated source SHA-256:', source_digest)

## Build the frozen OnHW adapter

Construction uses the predeclared 59-character task alphabet only. It never reads `train.json`, `val.json`, OnHW frequencies, or recognition results. `Ä` and `Ü` are retained as single-character fallbacks; no OnHW-derived bigrams are invented.

In [ ]:
adapter = build_frozen_onhw_adapter(SOURCE_PATH)
adapter_bytes = canonical_json_bytes(adapter)
adapter_digest = hashlib.sha256(adapter_bytes).hexdigest()

assert adapter['model_version'] == ADAPTER_POLICY_ID
assert adapter['size'] == ADAPTER_SIZE == 419
assert adapter['eligible_bigram_count'] == ADAPTER_BIGRAM_COUNT == 359
assert adapter['blank_id'] == 0
assert adapter['vocab'][''] == 0
assert adapter['adapter']['annotation_files_read'] is False
assert adapter_digest == ADAPTER_SHA256

summary = {
    'policy_id': adapter['model_version'],
    'classes_including_blank': adapter['size'],
    'single_characters': 59,
    'handwriting_bigrams': adapter['eligible_bigram_count'],
    'fallback_characters': adapter['adapter']['fallback_characters_absent_from_source'],
    'source_sha256': source_digest,
    'adapter_sha256': adapter_digest,
}
summary

## Verify or write the canonical artifact

Leave `WRITE_CANONICAL_ARTIFACT = False` during ordinary use. If the artifact is absent, or intentional reproduction is required, set it to `True`. Either path must produce bytes identical to the frozen checksum.

In [ ]:
if WRITE_CANONICAL_ARTIFACT:
    written_digest = write_adapter(adapter, OUTPUT_PATH)
    assert written_digest == ADAPTER_SHA256
    print('Wrote canonical artifact:', OUTPUT_PATH)
else:
    assert OUTPUT_PATH.is_file(), f'Canonical artifact not found: {OUTPUT_PATH}'
    committed_bytes = OUTPUT_PATH.read_bytes()
    assert committed_bytes == adapter_bytes
    print('Canonical artifact is byte-identical:', OUTPUT_PATH)

print('Canonical SHA-256:', ADAPTER_SHA256)

## Load through TVA and inspect segmentation

The production runtime preserves NFC normalization and DTLR's maximum-total-utility non-overlapping dynamic program. It is deliberately not TVA's legacy greedy Bigram tokenizer.

In [ ]:
tokenizer = get_tokenizer('handwriting_bigram')
tokenizer.load(OUTPUT_PATH)
assert tokenizer.size == 419

for word in ['Dabei', 'gerade', 'Küche', 'Übung']:
    result = tokenizer.segment(word)
    ids = tokenizer.encode(word)
    assert tokenizer.decode(ids) == unicodedata.normalize('NFC', word)
    print(word, '->', result['tokens'], '->', ids)

## Post-construction OnHW compatibility audit

This step reads labels only **after** the artifact has been built and authenticated. It checks compatibility; it does not select tokens, utilities, thresholds, or hyperparameters. Every WD/WI train and validation label must encode without blank IDs and round-trip to its NFC-normalized text.

In [ ]:
DATASETS = [
    REPO_ROOT / 'data/tva/onhw_words500_wd_word_rh',
    REPO_ROOT / 'data/tva/onhw_words500_wi_word_rh',
]

audit_rows = []
total_instances = 0
for dataset_dir in DATASETS:
    for split in ('train', 'val'):
        annotation_path = dataset_dir / f'{split}.json'
        with annotation_path.open(encoding='utf-8') as file:
            document = json.load(file)
        for fold, annotations in sorted(document['annotations'].items(), key=lambda item: int(item[0])):
            for annotation in annotations:
                label = unicodedata.normalize('NFC', annotation['label'])
                ids = tokenizer.encode(label)
                assert ids
                assert 0 not in ids
                assert all(1 <= index < tokenizer.size for index in ids)
                assert tokenizer.decode(ids) == label
            count = len(annotations)
            total_instances += count
            audit_rows.append({
                'dataset': dataset_dir.name,
                'split': split,
                'fold': int(fold),
                'labels_checked': count,
            })

assert total_instances == 251_990
print(f'Passed {len(audit_rows)} fold/split checks over {total_instances:,} label instances.')
audit_rows

## Reproducibility record

The expected source checksum is `5c5d9f1689a4802fc5e9451e5afe8abdfb090562587ab78ddd343211feb94dd4`. The expected canonical adapter checksum is `12ce25d8bedc552e6b3497ffb1d07e506b01b34296cc21b970f82a550cbf2bfe`. Use the same 419-class artifact unchanged for every WD and WI fold.